# Video Repetition Benchmark Results

Run this notebook with the project root as the Jupyter working directory. It discovers `results/**/metrics_by_seed.csv` automatically, then renders one section per benchmark dataset. Percentage values are displayed with `%` and are not multiplied again.

The benchmark MAE follows *Every Shot Counts*: average the per-video absolute error divided by that video's positive ground-truth count. `LegacyMAE` is the unnormalized mean absolute count error, reported in repetitions.

## 1. Import Libraries and Define Inputs

The example counts below are used to test the metric independently of benchmark files.

In [10]:
# 1. Import Libraries and Define Inputs
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# 2. Results Browser

Each dataset gets an independent leaderboard. Experiment settings (split, model, prompt, FPS, temperature) remain separate. Scoreboards average over seeds and show mean ± standard deviation; MAE, OBZ, and OBO are displayed as percentages, without another scaling step. LegacyMAE and RMSE are reported in repetition counts. An empty dataset section means its metrics files have not been generated yet.

In [11]:
DATASET_SECTIONS = ["Countix", "RepCount", "UCFRep", "OVR-Kinetics", "OVR-Ego4d"]
SETTING_COLUMNS = ["annotation_split", "model_name", "prompt_name", "fps", "temperature"]
METRIC_COLUMNS = ["mae_percent", "legacy_mae_count", "rmse_count", "obz_percent", "obo_percent"]
PERCENT_METRICS = {"mae_percent", "obz_percent", "obo_percent"}


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root with pyproject.toml and results/")


PROJECT_ROOT = find_project_root()
RESULTS_ROOT = PROJECT_ROOT / "results"


def load_metric_runs(results_root=RESULTS_ROOT):
    paths = sorted(results_root.rglob("metrics_by_seed.csv"))
    frames = []
    failures = []
    for path in paths:
        try:
            frame = pd.read_csv(path)
            if frame.empty:
                continue
            frame["_source_file"] = str(path.relative_to(results_root))
            frames.append(frame)
        except (OSError, pd.errors.ParserError, UnicodeDecodeError) as exc:
            failures.append(f"{path}: {exc}")
    if failures:
        print("Skipped unreadable metric files:")
        for failure in failures:
            print(f"- {failure}")
    if not frames:
        return pd.DataFrame()
    data = pd.concat(frames, ignore_index=True)
    legacy_columns = {
        "nmae_n": "mae_n",
        "nmae_percent": "mae_percent",
        "mae_count": "legacy_mae_count",
    }
    for old_column, new_column in legacy_columns.items():
        if old_column in data:
            if new_column in data:
                data[new_column] = data[new_column].combine_first(data[old_column])
                data = data.drop(columns=old_column)
            else:
                data = data.rename(columns={old_column: new_column})
    numeric_columns = [
        "fps", "temperature", "seed", "n_total", "n_valid_predictions", "mae_n", *METRIC_COLUMNS
    ]
    for column in numeric_columns:
        if column in data:
            data[column] = pd.to_numeric(data[column], errors="coerce")
    return data


metric_runs = load_metric_runs()
print(f"Loaded {len(metric_runs):,} per-seed metric rows from {RESULTS_ROOT}")

Loaded 15 per-seed metric rows from /mnt/Workspace/audiovisual-video-repetition-counting/results


In [12]:
def _mean_sd_label(mean, sd, suffix=""):
    if pd.isna(mean):
        return "n/a"
    spread = "n/a" if pd.isna(sd) else f"{sd:.2f}"
    return f"{mean:.2f} ± {spread}{suffix}"


def _style_table(styler):
    return styler.set_table_styles(
        [
            {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "1.05em"), ("font-weight", "600"), ("text-align", "left")]},
            {"selector": "th", "props": [("background-color", "#193b3b"), ("color", "white"), ("padding", "7px 10px")]},
            {"selector": "td", "props": [("padding", "6px 10px"), ("border-bottom", "1px solid #dce5e3")]},
            {"selector": "tbody tr:hover", "props": [("background-color", "#edf5f2")]},
        ]
    ).set_properties(**{"text-align": "right", "white-space": "nowrap"})


def show_dataset_results(dataset_name, data=metric_runs):
    print(f"{dataset_name}")
    if data.empty or "dataset" not in data:
        display(Markdown("*No results available yet.*"))
        return
    dataset_data = data[data["dataset"].astype(str).str.casefold() == dataset_name.casefold()].copy()
    if dataset_data.empty:
        display(Markdown("*No results available yet.*"))
        return

    required_columns = [
        *SETTING_COLUMNS, "seed", "n_total", "n_valid_predictions", "mae_n", *METRIC_COLUMNS
    ]
    missing = [column for column in required_columns if column not in dataset_data]
    if missing:
        display(Markdown(f"*Metric file is missing required columns: {', '.join(missing)}.*"))
        return

    dataset_data["coverage_percent"] = np.where(
        dataset_data["n_total"] > 0,
        100.0 * dataset_data["n_valid_predictions"] / dataset_data["n_total"],
        np.nan,
    )
    grouping = [*SETTING_COLUMNS]
    grouped = dataset_data.groupby(grouping, dropna=False)
    summary = grouped.agg(
        seeds=("seed", "nunique"),
        coverage_mean=("coverage_percent", "mean"),
        coverage_sd=("coverage_percent", "std"),
        **{f"{metric}_mean": (metric, "mean") for metric in METRIC_COLUMNS},
        **{f"{metric}_sd": (metric, "std") for metric in METRIC_COLUMNS},
    ).reset_index()
    summary = summary.sort_values(["mae_percent_mean", "legacy_mae_count_mean"], na_position="last")

    scoreboard = summary[grouping + ["seeds"]].copy()
    scoreboard["Coverage (mean ± SD, %)"] = summary.apply(
        lambda row: _mean_sd_label(row["coverage_mean"], row["coverage_sd"], "%"), axis=1
    )
    for metric, label in [
        ("mae_percent", "MAE (mean ± SD, %)"),
        ("legacy_mae_count", "LegacyMAE (mean ± SD, count)"),
        ("rmse_count", "RMSE (mean ± SD, count)"),
        ("obz_percent", "Exact / OBZ (mean ± SD, %)"),
        ("obo_percent", "Within-one / OBO (mean ± SD, %)"),
    ]:
        suffix = "%" if metric in PERCENT_METRICS else ""
        scoreboard[label] = summary.apply(
            lambda row, metric=metric, suffix=suffix: _mean_sd_label(
                row[f"{metric}_mean"], row[f"{metric}_sd"], suffix
            ),
            axis=1,
        )

    display(
        _style_table(scoreboard.style)
        .hide(axis="index")
        .set_caption(f"{dataset_name} · experiment scoreboard (lower MAE is better)")
    )

    detail_columns = [
        *grouping, "seed", "run_status", "n_total", "n_valid_predictions", "mae_n", *METRIC_COLUMNS
    ]
    per_seed = dataset_data[detail_columns].sort_values(
        [*grouping, "seed"], na_position="last"
    ).reset_index(drop=True)
    formats = {
        "fps": "{:.2f}",
        "temperature": "{:.2f}",
        "seed": "{:.0f}",
        "n_total": "{:.0f}",
        "n_valid_predictions": "{:.0f}",
        "mae_n": "{:.0f}",
        "mae_percent": "{:.2f}%",
        "legacy_mae_count": "{:.2f}",
        "rmse_count": "{:.2f}",
        "obz_percent": "{:.2f}%",
        "obo_percent": "{:.2f}%",
    }
    display(
        _style_table(per_seed.style)
        .hide(axis="index")
        .format(formats, na_rep="n/a")
        .set_caption(f"{dataset_name} · per-seed results (percentage values are stored percentage points)")
    )


for dataset_section in DATASET_SECTIONS:
    display(Markdown(f"## {dataset_section}"))
    show_dataset_results(dataset_section)

## Countix

Countix


*No results available yet.*

## RepCount

RepCount


...

OVR-Kinetics


*No results available yet.*

## OVR-Ego4d

OVR-Ego4d


*No results available yet.*

### Optional Prediction Drilldown

Prediction CSVs can be much larger than metric summaries. The helper below reads them only when explicitly called and selects columns that exclude `raw_response`; it shows non-OK rows and the largest absolute count errors for one dataset.

In [13]:
PREDICTION_DETAIL_COLUMNS = {
    "dataset", "annotation_split", "annotation_id", "video_name", "gt_count",
    "pred_count", "abs_error", "status", "error_message", "seed", "model_name",
    "prompt_name", "fps", "temperature",
}


def show_prediction_drilldown(dataset_name, top_n=20, results_root=RESULTS_ROOT):
    paths = sorted(results_root.rglob("predictions.csv"))
    frames = []
    for path in paths:
        try:
            frame = pd.read_csv(
                path,
                usecols=lambda column: column in PREDICTION_DETAIL_COLUMNS,
                low_memory=False,
            )
        except (OSError, pd.errors.ParserError, UnicodeDecodeError, ValueError) as exc:
            print(f"Skipped {path}: {exc}")
            continue
        if "dataset" in frame and not frame.empty:
            frame = frame[frame["dataset"].astype(str).str.casefold() == dataset_name.casefold()]
            if not frame.empty:
                frame["_source_file"] = str(path.relative_to(results_root))
                frames.append(frame)
    if not frames:
        display(Markdown(f"*No prediction rows available for {dataset_name}.*"))
        return

    predictions = pd.concat(frames, ignore_index=True)
    if "abs_error" in predictions:
        predictions["abs_error"] = pd.to_numeric(predictions["abs_error"], errors="coerce")
    error_rows = predictions[predictions["status"] != "ok"].copy()
    worst_rows = predictions[predictions["abs_error"].notna()].nlargest(top_n, "abs_error")
    columns = [
        column for column in [
            "annotation_split", "video_name", "gt_count", "pred_count", "abs_error",
            "status", "error_message", "seed", "model_name", "prompt_name",
            "fps", "temperature", "_source_file",
        ] if column in predictions
    ]
    print(f"{dataset_name}: {len(error_rows):,} non-OK prediction rows")
    if not error_rows.empty:
        display(_style_table(error_rows[columns].head(top_n).style.hide(axis="index")))
    display(
        _style_table(worst_rows[columns].style.hide(axis="index"))
        .set_caption(f"{dataset_name} · largest absolute count errors")
    )


# Example: 
show_prediction_drilldown("RepCount", top_n=20)

RepCount: 336 non-OK prediction rows


annotation_split,video_name,gt_count,pred_count,abs_error,status,error_message,seed,model_name,prompt_name,fps,temperature,_source_file
test,stu9_48.mp4,27.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (12558) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",13,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu9_48.mp4,27.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (12558) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",17,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu9_48.mp4,27.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (12558) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",23,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu9_48.mp4,27.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (12558) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",42,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu9_48.mp4,27.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (12558) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",99,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu8_70.mp4,4.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (11963) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",13,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu8_70.mp4,4.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (11963) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",17,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu8_70.mp4,4.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (11963) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",23,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu8_70.mp4,4.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (11963) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",42,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu8_70.mp4,4.000000,nan,nan,inference_error,"Error code: 400 - {'error': {'message': ""Input length (11963) exceeds model's maximum context length (8192)."", 'type': 'BadRequestError', 'param': None, 'code': 400}}",99,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv


annotation_split,video_name,gt_count,pred_count,abs_error,status,error_message,seed,model_name,prompt_name,fps,temperature,_source_file
test,train2737.mp4,3.000000,100.000000,97.000000,ok,nan,23,Qwen3-VL-8B-Instruct,Count Class Action,2.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-2/temperature-0.2/predictions.csv
test,train2737.mp4,3.000000,100.000000,97.000000,ok,nan,42,Qwen3-VL-8B-Instruct,Count Class Action,2.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-2/temperature-0.2/predictions.csv
test,train2737.mp4,3.000000,100.000000,97.000000,ok,nan,13,Qwen3-VL-8B-Instruct,Count Repetitions,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-repetitions/fps-1/temperature-0.2/predictions.csv
test,train2737.mp4,3.000000,100.000000,97.000000,ok,nan,17,Qwen3-VL-8B-Instruct,Count Repetitions,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-repetitions/fps-1/temperature-0.2/predictions.csv
test,train2737.mp4,3.000000,100.000000,97.000000,ok,nan,23,Qwen3-VL-8B-Instruct,Count Repetitions,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-repetitions/fps-1/temperature-0.2/predictions.csv
test,train2737.mp4,3.000000,100.000000,97.000000,ok,nan,42,Qwen3-VL-8B-Instruct,Count Repetitions,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-repetitions/fps-1/temperature-0.2/predictions.csv
test,train2737.mp4,3.000000,100.000000,97.000000,ok,nan,99,Qwen3-VL-8B-Instruct,Count Repetitions,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-repetitions/fps-1/temperature-0.2/predictions.csv
test,stu3_31.mp4,92.000000,0.000000,92.000000,ok,nan,13,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu3_31.mp4,92.000000,0.000000,92.000000,ok,nan,17,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
test,stu3_31.mp4,92.000000,0.000000,92.000000,ok,nan,23,Qwen3-VL-8B-Instruct,Count Class Action,1.000000,0.200000,repcount/qwen3-vl-8b/video-only/count-class-action/fps-1/temperature-0.2/predictions.csv
